## Imports

In [8]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
## Parameter 
start_date = '20240318'
end_date = '20240331'

In [5]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)
local_datasets = '/Users/rapido/local-datasets/customer/appography/appography_v1/city/jaipur/'
print(local_datasets)

/Users/rapido/analytics/customer/Appography/Appography_v1
/Users/rapido/local-datasets/customer/appography/appography_v1/city/jaipur/


In [6]:
##usecase_tag

usecase_tag_query = f"""

WITH active_customers AS (

    SELECT 
        customer_id,
        order_id,
        drop_location_hex_10 hex_10
    FROM 
        orders.order_logs_snapshot
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND channel_host = 'android'
        AND city_name = 'Jaipur'
    GROUP BY 1,2,3
    ), 

    use_case AS (
    
    SELECT
        hex_10, 
        usecase_tag
    FROM
        (
        SELECT 
            hex_10, 
            combined_final_usecase_accuracy as usecase_tag,
            ROW_NUMBER() OVER(PARTITION BY hex_10 ORDER BY run_date DESC) seq_no
        FROM
            hive.experiments_internal.combined_usecase_hex_10_level
        WHERE 
            aoi = 'Bangalore, India'
        )
    WHERE   
        seq_no = 1
    ),
    
    merge AS (
    SELECT
        customer_id,
        usecase_tag,
        ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY orders DESC) seq_no
    FROM 
        (
        SELECT
            a.customer_id,
            COALESCE(b.usecase_tag, 'Unknown') usecase_tag,
            COUNT(DISTINCT order_id) orders
        FROM 
            active_customers a
        LEFT JOIN 
            use_case b
            ON a.hex_10 = b.hex_10
        GROUP BY 1,2
        )
    WHERE 
        usecase_tag != 'Unknown'
    )
    
    SELECT
        a.customer_id,
        COALESCE(b.usecase_tag, 'Unknown') usecase_tag
    FROM 
        active_customers a
    LEFT JOIN 
        merge b 
        ON a.customer_id = b.customer_id
        AND seq_no = 1
    GROUP BY 1,2

"""

df_usecase_tag_query = pd.read_sql(usecase_tag_query, connection)
df_usecase_tag_query.to_csv(local_datasets + 'raw/usecase_tag_v1.csv', index=False)

In [7]:
df_usecase_tag = pd.read_csv(local_datasets + 'raw/usecase_tag_v1.csv')
print(df_usecase_tag.shape)
df_usecase_tag.head(2)

(250719, 2)


,customer_id,usecase_tag
0,622ebf758bbc852a80477c2c,Unknown
1,62acb711279b188793ec31b1,Unknown


### Active customer (RR-customers)

In [9]:
df_bangalore_active_customer = pd.read_csv(local_datasets + 'raw/jaipur_customers_v1.csv')
df_bangalore_active_customer = df_bangalore_active_customer.drop('app_list_set', axis=1)
print(df_bangalore_active_customer.shape)
df_bangalore_active_customer.head(2)

(220564, 16)


,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc
0,6495a36f0605e65112d4434a,MALE,289.0,NaN,8.0,3.0,2.0,PHH,8,HOOK,MEDIUM_INCOME,ONLY_LINK,UNKNOWN,NPS,2.0,c. medium rpc
1,5cdcfd9425ee3218d4c85497,MALE,1788.0,37.0,7.0,5.0,4.0,PHH,30,HOOK,MEDIUM_INCOME,ONLY_LINK,SHORT_DISTANCE,PS,4.0,d. high rpc


In [10]:
# df_bangalore_active_customer = pd.merge(df_bangalore_active_customer, df_usecase_tag,
#                                         how='left', on=['customer_id'])
df_bangalore_active_customer = df_bangalore_active_customer

### customer installed apps

In [11]:
df_customer_mapped = pd.read_csv(local_datasets + 'processed/customer_app_categories_mapped_v1.csv')
print(df_customer_mapped.shape)
df_customer_mapped.head(2)

(220347, 14)


,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Office,Finance_Investment,Ride haling,Rest
0,573f28f49b0ffc283676f3c1,"['telegram', 'instagram', 'cred', 'zoom', 'ama...",42,"['Tools', 'Social', 'Delivery_Food', 'OTT', 'S...",13,"['Finance_Investment', 'Ride haling', 'Office'...",4,0,0,0,1,1,1,1
1,573f28f49b0ffc283676f74c,"['telegram', 'instagram', 'cred', 'zomato', 'l...",26,"['Tools', 'Social', 'Delivery_Food', 'Gaming',...",11,"['Finance_Investment', 'Gaming', 'Ride haling'...",4,0,0,1,0,1,1,1


### Customer app & cat explode mapping

In [12]:
df_customer_app_cat_mapping = pd.read_csv(local_datasets + 'processed/customer_app_categories_mapping_v1.csv')
df_customer_cat_mapping = df_customer_app_cat_mapping[['customer_id', 
#                                                        'categories_l1', 
                                                       'categories_l2']]\
                            .drop_duplicates()
df_customer_cat_mapping.head(1)

,customer_id,categories_l2
0,6495a36f0605e65112d4434a,Gaming


In [13]:
total_customers = df_customer_cat_mapping.customer_id.nunique()
total_customers

220347

In [14]:
print(df_customer_app_cat_mapping['categories_l1'].unique())

['Gaming' 'Office' 'Social' 'Tools' 'Delivery_Food' 'Travel_Ridehailing'
 'Travel_Bookings' 'News' 'Ecommerce' 'Finance_Transactions' 'OTT'
 'Streaming_Music' 'Finance_Investment' 'Educational' 'Entertainment'
 'Delivery_Grocery' 'Health' 'Wearable' 'Driver_App' 'Fantasy_Sports'
 'Dating' 'Finance_News' 'Devotional']


In [15]:
print(df_customer_app_cat_mapping['categories_l2'].unique())

['Gaming' 'Office' 'Rest' 'Ride haling' 'Finance_Investment' 'Educational'
 'Driver_App']


In [16]:
# single_category = ['Travel_Ridehailing']

# ### Office
# print(single_category)
# df_temp = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l1'].isin(single_category)]\
#             .groupby(['categories_l1','app_list'])\
#             .agg(customers = ('customer_id','nunique'))\
#             .sort_values(['customers'],ascending=False)\
#             .reset_index()
# df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
# df_temp

In [17]:
### Office
df_temp = df_customer_app_cat_mapping[~df_customer_app_cat_mapping['categories_l2'].isin(['Rest'])]\
            .groupby(['categories_l2','app_list_set'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(by=['categories_l2', 'customers'], ascending=[False, False])\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list_set,customers,%
0,Ride haling,uber,96187,43.65
1,Ride haling,ola,89001,40.39
2,Ride haling,in drive,10533,4.78
3,Ride haling,jugnoo,3593,1.63
4,Ride haling,blusmart,839,0.38
5,Ride haling,namma yatri,356,0.16
6,Ride haling,quick ride,169,0.08
7,Ride haling,driveu driver app,22,0.01
8,Ride haling,quickride,2,0.00
9,Office,onedrive,40659,18.45


### Merge raw data

In [18]:
df_bangalore_active_customer

,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc
0,6495a36f0605e65112d4434a,MALE,289.0,NaN,8.0,3.0,2.0,PHH,8,HOOK,MEDIUM_INCOME,ONLY_LINK,UNKNOWN,NPS,2.0,c. medium rpc
1,5cdcfd9425ee3218d4c85497,MALE,1788.0,37.0,7.0,5.0,4.0,PHH,30,HOOK,MEDIUM_INCOME,ONLY_LINK,SHORT_DISTANCE,PS,4.0,d. high rpc
2,6608cccadb0e9c93551a5d08,MALE,7.0,NaN,2.0,1.0,1.0,HH,1,HANDHOLDING,HIGH_INCOME,ONLY_LINK,LONG_DISTANCE,UNKNOWN,1.0,b. low rpc
3,648e8473c189b9c8fd087193,MALE,294.0,NaN,8.0,1.0,1.0,PHH,6,HOOK,MEDIUM_INCOME,ONLY_AUTO,UNKNOWN,PS,1.0,b. low rpc
4,5f6700970a97c86130d4c31d,MALE,1295.0,34.0,8.0,2.0,2.0,PHH,28,SUSTENANCE,HIGH_INCOME,ONLY_LINK,LONG_DISTANCE,PS,2.0,c. medium rpc
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220559,62d6c32446b3881f1bb19a14,MALE,628.0,23.0,3.0,1.0,0.0,HH,4,INACTIVE,UNKNOWN,ONLY_AUTO,UNKNOWN,UNKNOWN,NaN,a. zero rpc
220560,65c041aea3898724923547e4,MALE,62.0,NaN,2.0,1.0,1.0,HH,1,HANDHOLDING,MEDIUM_INCOME,ONLY_AUTO,UNKNOWN,UNKNOWN,1.0,b. low rpc
220561,6303a56751f714ef60630377,FEMALE,594.0,NaN,7.0,1.0,1.0,PHH,22,HOOK,MEDIUM_INCOME,LINK_AUTO,SHORT_DISTANCE,UNKNOWN,1.0,b. low rpc
220562,636084bbd58af71099dee9c1,MALE,523.0,NaN,14.0,1.0,1.0,PHH,6,HOOK,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,1.0,b. low rpc


In [19]:
df_bangalore_active_customer[df_bangalore_active_customer['customer_id'].isin(['65e4c108d4f28464de89d5b0'])]

,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc


In [20]:
df_customer_mapped

,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Office,Finance_Investment,Ride haling,Rest
0,573f28f49b0ffc283676f3c1,"['telegram', 'instagram', 'cred', 'zoom', 'ama...",42,"['Tools', 'Social', 'Delivery_Food', 'OTT', 'S...",13,"['Finance_Investment', 'Ride haling', 'Office'...",4,0,0,0,1,1,1,1
1,573f28f49b0ffc283676f74c,"['telegram', 'instagram', 'cred', 'zomato', 'l...",26,"['Tools', 'Social', 'Delivery_Food', 'Gaming',...",11,"['Finance_Investment', 'Gaming', 'Ride haling'...",4,0,0,1,0,1,1,1
2,573f28f79b0ffc28367708b0,"['telegram', 'ludo king', 'zoom', 'cred', 'zom...",40,"['Tools', 'Fantasy_Sports', 'Social', 'News', ...",16,"['Gaming', 'Finance_Investment', 'Office', 'Ri...",5,0,0,1,1,1,1,1
3,573f29099b0ffc28367741d9,"['telegram', 'okcupid', 'instagram', 'zoom', '...",41,"['Tools', 'Social', 'News', 'Delivery_Food', '...",15,"['Gaming', 'Ride haling', 'Office', 'Rest']",4,0,0,1,1,0,1,1
4,573f29099b0ffc283677437d,"['telegram', 'instagram', 'cred', 'zoom', 'zom...",42,"['Tools', 'Social', 'Delivery_Food', 'OTT', 'S...",13,"['Finance_Investment', 'Ride haling', 'Office'...",4,0,0,0,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220342,6609a5f275d2876d82f35aaf,"['instagram', 'snapchat', 'netflix', 'spotify']",4,"['Social', 'OTT', 'Streaming_Music']",3,['Rest'],1,0,0,0,0,0,0,1
220343,6609a64b8fcbab1e8002778f,"['google sheets', 'instagram', 'facebook', 'fl...",17,"['Tools', 'Social', 'Gaming', 'OTT', 'Streamin...",7,"['Gaming', 'Ride haling', 'Rest']",3,0,0,1,0,0,1,1
220344,6609a660c1792a54fc022554,"['telegram', 'ludo king', 'instagram', 'zoom',...",10,"['Social', 'News', 'Delivery_Food', 'Gaming', ...",7,"['Gaming', 'Office', 'Rest']",3,0,0,1,1,0,0,1
220345,6609a681a9381b9cc63a9246,"['angel broking', 'instagram', 'facebook', 'go...",13,"['Tools', 'Social', 'News', 'Gaming', 'OTT', '...",9,"['Gaming', 'Finance_Investment', 'Rest']",3,0,0,1,0,1,0,1


In [21]:
df_customer_data = pd.merge(df_bangalore_active_customer, df_customer_mapped,
                            how='inner', on=['customer_id']
                           )
print(df_customer_data.shape)
df_customer_data.head(1)

(220347, 29)


,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Office,Finance_Investment,Ride haling,Rest
0,6495a36f0605e65112d4434a,MALE,289.0,NaN,8.0,3.0,2.0,PHH,8,HOOK,MEDIUM_INCOME,ONLY_LINK,UNKNOWN,NPS,2.0,c. medium rpc,"['onedrive', 'instagram', 'microsoft 365', 'fa...",10,"['Tools', 'Social', 'Gaming', 'Office']",4,"['Gaming', 'Office', 'Rest']",3,0,0,1,1,0,0,1


#### Derived columns

In [22]:
## RPC

df_customer_data['rpc'] =  df_customer_data['net_count'].replace(0, np.nan)
df_customer_data.rpc.describe()

count    171080.000000
mean          2.137123
std           2.364220
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          73.000000
Name: rpc, dtype: float64

In [23]:
df_customer_data['rpc_bucket'] = np.where(df_customer_data['net_count'] < 1 , 'a. ZERO',
                                    np.where(df_customer_data['net_count'] < 2 , 'b. LOW',
                                        np.where(df_customer_data['net_count'] < 4 , 'c. MEDIUM', 
                                            np.where(df_customer_data['net_count'] >= 4 , 'd. HIGH', None))))

In [24]:
## app_count_bucket

df_customer_data.app_count.describe()

count    220347.000000
mean         15.033456
std           7.706327
min           1.000000
25%           9.000000
50%          14.000000
75%          19.000000
max          72.000000
Name: app_count, dtype: float64

In [25]:
df_customer_data['app_count_bucket'] = np.where(df_customer_data['net_count'] < 5 , '1-5',
                                        np.where(df_customer_data['net_count'] < 10 , '6-10',
                                          np.where(df_customer_data['net_count'] < 16 , '11-15', 'Above 15')))

In [26]:
## category_count_bucket

df_customer_data['category_count'] =  df_customer_data['categories_l2_count'].replace(0, 1)
df_customer_data.category_count.describe()

count    220347.000000
mean          2.759729
std           1.232630
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max           7.000000
Name: category_count, dtype: float64

In [27]:
df_customer_data['category_count_bucket'] = np.where(df_customer_data['category_count'] < 2 , '1',
                                              np.where(df_customer_data['category_count'] < 3 , '2',
                                                np.where(df_customer_data['category_count'] < 4 , '3','Above 3')))

In [28]:
## category_count

# df_customer_data['category_count'] =  df_customer_data['categories_l2_count'].replace(0, 1)
df_customer_data.rapido_age.describe()

count    220329.000000
mean        699.796990
std         565.516005
min           3.000000
25%         221.000000
50%         606.000000
75%         994.000000
max        2875.000000
Name: rapido_age, dtype: float64

#### Merge

In [29]:
df_customer_data_explode = pd.merge(df_customer_data,
                                    df_customer_cat_mapping,
                                    how='inner', on =['customer_id'])

df_customer_data_explode[df_customer_data_explode['customer_id'] == '6475f56aba9a99b915ccc086'].head()

,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc,app_list,app_count,categories_l1,categories_l1_count,categories_l2_x,categories_l2_count,Driver_App,Educational,Gaming,Office,Finance_Investment,Ride haling,Rest,rpc_bucket,app_count_bucket,category_count,category_count_bucket,categories_l2_y


## User Base Distribution analysis

LTR-Segment

Service Affinity

Distance Affinity

Customers Use Case

AO- Net Funnel

Gender

Age (Whatever fill rate we have)

Income

RPC

In [30]:
tot_customers = df_customer_cat_mapping.customer_id.nunique()
df_category_agg = df_customer_cat_mapping\
                    .groupby(['categories_l2'])\
                    .agg(total_customers = ('customer_id','nunique'))\
                    .sort_values(['total_customers'], ascending=False)\
                    .reset_index()
df_category_agg['% distribution'] =  (df_category_agg['total_customers']*100.00/tot_customers).round(2)
df_category_agg

,categories_l2,total_customers,% distribution
0,Rest,220259,99.96
1,Ride haling,133655,60.66
2,Office,84756,38.46
3,Finance_Investment,60253,27.34
4,Gaming,56308,25.55
5,Educational,37358,16.95
6,Driver_App,15509,7.04


#### LTR-Segment

In [31]:
ltr_segment_city = df_customer_data\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
ltr_segment_city['% city threshold'] =  (ltr_segment_city['customers']*100.00/ltr_segment_city.customers.sum()).round(2)
ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,56238,25.52
1,LTR 0,9179,4.17
2,PHH,154859,70.28
3,UNKNOWN,71,0.03


In [32]:
df1 = df_customer_data_explode[~df_customer_data_explode['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customers'])

customers                                                       \
categories_l2 Driver_App Educational Finance_Investment Gaming Office    Rest   
ltr_segment                                                                     
HH                  3936        8496              12262  14210  17635   56205   
LTR 0                823        1275               1657   2282   2532    9173   
PHH                10744       27576              46312  39802  64547  154810   

                           
categories_l2 Ride haling  
ltr_segment                
HH                  26492  
LTR 0                3918  
PHH                103196

In [33]:
# print('% Distribution of customers across individual categories.')
# df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customer %'])

In [34]:
print('% Distribution of customers across individual Segment.')
df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customers %'])

% Distribution of customers across individual Segment.


customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
ltr_segment                                                                     
HH                   7.00       15.11              21.80  25.27  31.36  99.94   
LTR 0                8.97       13.89              18.05  24.86  27.58  99.93   
PHH                  6.94       17.81              29.91  25.70  41.68  99.97   

                           
categories_l2 Ride haling  
ltr_segment                
HH                  47.11  
LTR 0               42.68  
PHH                 66.64

#### Other testing

In [35]:
df_test = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l2'].isin(['Office', 
                                                                                         'Educational',
                                                                                         'Ride haling'
                                                                                        ])]
df_test = pd.merge(df_test,df_bangalore_active_customer[['customer_id', 'ltr_segment']], how='inner', on='customer_id')
df_office = df_test[df_test['categories_l2'].isin(['Office'])]
df_education = df_test[df_test['categories_l2'].isin(['Educational'])]
df_ride_hailing = df_test[df_test['categories_l2'].isin(['Ride haling'])]

##### Explode - Ride Hailing

In [36]:
rh_ltr_segment_city = df_ride_hailing\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
rh_ltr_segment_city['% city threshold'] =  (rh_ltr_segment_city['customers']*100.00/rh_ltr_segment_city.customers.sum()).round(2)
rh_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,26492,19.82
1,LTR 0,3918,2.93
2,PHH,103196,77.21
3,UNKNOWN,49,0.04


In [37]:
df1 = df_ride_hailing[~df_ride_hailing['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
ride_hailing_total_customers = df_ride_hailing.customer_id.nunique()
df2 = pd.merge(df1,rh_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/ride_hailing_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                          \
app_name     blusmart driveu driver app in drive  jugnoo namma yatri      ola   
ltr_segment                                                                     
HH               85.0               3.0   1698.0   560.0        23.0  16680.0   
LTR 0            15.0               1.0    249.0    55.0         4.0   2490.0   
PHH             739.0              18.0   8583.0  2977.0       329.0  69801.0   

                                           
app_name    quick ride quickride     uber  
ltr_segment                                
HH                11.0       NaN  17755.0  
LTR 0              2.0       NaN   2525.0  
PHH              156.0       2.0  75870.0

In [38]:
# print('% Distribution of customers across individual apps.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [39]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual segment.


customers %                                                       \
app_name       blusmart driveu driver app in drive jugnoo namma yatri    ola   
ltr_segment                                                                    
HH                 0.32              0.01     6.41   2.11        0.09  62.96   
LTR 0              0.38              0.03     6.36   1.40        0.10  63.55   
PHH                0.72              0.02     8.32   2.88        0.32  67.64   

                                         
app_name    quick ride quickride   uber  
ltr_segment                              
HH                0.04       NaN  67.02  
LTR 0             0.05       NaN  64.45  
PHH               0.15       0.0  73.52

##### Explode - Office

In [40]:
df_office.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment
0,6495a36f0605e65112d4434a,outlook,outlook,Office,Office,PHH
1,6495a36f0605e65112d4434a,onedrive,onedrive,Office,Office,PHH


In [41]:
def assign_office_value(row):
    if row['app_name'] in ['asana','cisco','confluence','github','google analytics',
                           'intune  company portal','jira','miro','slack','trello','zoho mail','zoho meeting']:
        return 'code_office_app'
    else:
        return 'secondary_office_app'

df_office['app_name_tag'] = df_office.apply(assign_office_value, axis=1)
df_office.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment,app_name_tag
0,6495a36f0605e65112d4434a,outlook,outlook,Office,Office,PHH,secondary_office_app
1,6495a36f0605e65112d4434a,onedrive,onedrive,Office,Office,PHH,secondary_office_app


In [42]:
df_office[df_office['app_name_tag'] == 'code_office_app'].app_name.unique()

array(['intune  company portal', 'trello', 'github', 'google analytics',
       'slack', 'zoho mail', 'jira', 'asana', 'zoho meeting', 'cisco',
       'miro', 'confluence'], dtype=object)

In [43]:
office_ltr_segment_city = df_office\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
office_ltr_segment_city['% city threshold'] =  (office_ltr_segment_city['customers']*100.00/office_ltr_segment_city.customers.sum()).round(2)
office_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,17635,20.81
1,LTR 0,2532,2.99
2,PHH,64547,76.16
3,UNKNOWN,42,0.05


In [44]:
df1 = df_office[~df_office['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
office_total_customers = df_office.customer_id.nunique()
df2 = pd.merge(df1,office_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/office_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                   \
app_name        asana cisco confluence dropbox github google analytics   
ltr_segment                                                              
HH                9.0   6.0        6.0   221.0  107.0             35.0   
LTR 0             1.0   1.0        NaN    31.0   11.0              5.0   
PHH              77.0  26.0       33.0   930.0  535.0            210.0   

                                                                              \
app_name    google authenticator intune  company portal   jira microsoft 365   
ltr_segment                                                                    
HH                        1115.0                  546.0   20.0        5693.0   
LTR 0                      123.0                   73.0    NaN         822.0   
PHH                       5569.0                 3425.0  172.0       22283.0   

                                                                              \
app_name    microsoft teams  miro ms authenticator nishtha onedrive  outlook   
ltr_segment                                                                    
HH                   3106.0   2.0            982.0     1.0   9121.0   4977.0   
LTR 0                 379.0   NaN            115.0     NaN   1387.0    692.0   
PHH                 14943.0  28.0           5788.0     1.0  30134.0  19723.0   

                                                                           
app_name    pocket   slack trello   webex zoho mail zoho meeting     zoom  
ltr_segment                                                                
HH             6.0   141.0   17.0   657.0      62.0         17.0   6758.0  
LTR 0          NaN    19.0    2.0    84.0       9.0          3.0    920.0  
PHH           55.0  1447.0  106.0  2794.0     400.0        114.0  27249.0

In [45]:
# print('% Distribution of customers across individual categories.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [46]:
print('% Distribution of customers across individual app.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual app.


customers %                                                   \
app_name          asana cisco confluence dropbox github google analytics   
ltr_segment                                                                
HH                 0.05  0.03       0.03    1.25   0.61             0.20   
LTR 0              0.04  0.04        NaN    1.22   0.43             0.20   
PHH                0.12  0.04       0.05    1.44   0.83             0.33   

                                                                             \
app_name    google authenticator intune  company portal  jira microsoft 365   
ltr_segment                                                                   
HH                          6.32                   3.10  0.11         32.28   
LTR 0                       4.86                   2.88   NaN         32.46   
PHH                         8.63                   5.31  0.27         34.52   

                                                                             \
app_name    microsoft teams  miro ms authenticator nishtha onedrive outlook   
ltr_segment                                                                   
HH                    17.61  0.01             5.57    0.01    51.72   28.22   
LTR 0                 14.97   NaN             4.54     NaN    54.78   27.33   
PHH                   23.15  0.04             8.97    0.00    46.69   30.56   

                                                                     
app_name    pocket slack trello webex zoho mail zoho meeting   zoom  
ltr_segment                                                          
HH            0.03  0.80   0.10  3.73      0.35         0.10  38.32  
LTR 0          NaN  0.75   0.08  3.32      0.36         0.12  36.33  
PHH           0.09  2.24   0.16  4.33      0.62         0.18  42.22

In [47]:
df1 = df_office[~df_office['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name_tag', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name_tag', 'customers'],ascending=[False,True])\
.reset_index()
office_total_customers = df_office.customer_id.nunique()
df2 = pd.merge(df1,office_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/office_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers'])

customers                     
app_name_tag code_office_app secondary_office_app
ltr_segment                                      
HH                       907                17521
LTR 0                    115                 2506
PHH                     5945                63886

In [48]:
#### print('% Distribution of customers across individual app.')
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers %'])

customers %                     
app_name_tag code_office_app secondary_office_app
ltr_segment                                      
HH                      5.14                99.35
LTR 0                   4.54                98.97
PHH                     9.21                98.98

##### Explode - Education

In [49]:
df_education.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment
12,648e8473c189b9c8fd087193,doubtnut,doubtnut,Educational,Educational,PHH
16,5f6700970a97c86130d4c31d,unacademy,unacademy,Educational,Educational,PHH


In [50]:
def assign_education_value(row):
    if row['app_name'] in ['aakash','byju','chegg study','diksha','e-pg pathshala',
                           'google classroom', 'microsoft math solver','moodle','mycbseguide', 
                           'physics wallah','simplilearn','vedantu','vidyakul'
                          ]:
#         brainly duolingo
        return 'code_education_app'
    else:
        return 'secondary_education_app'

df_education['app_name_tag'] = df_education.apply(assign_education_value, axis=1)
df_education.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment,app_name_tag
12,648e8473c189b9c8fd087193,doubtnut,doubtnut,Educational,Educational,PHH,secondary_education_app
16,5f6700970a97c86130d4c31d,unacademy,unacademy,Educational,Educational,PHH,secondary_education_app


In [51]:
df_education[df_education['app_name_tag'] == 'code_education_app'].app_name.unique()

array(['google classroom', 'physics wallah', 'moodle', 'byju', 'diksha',
       'e-pg pathshala', 'simplilearn', 'chegg study', 'mycbseguide',
       'vedantu', 'microsoft math solver', 'aakash', 'vidyakul'],
      dtype=object)

In [52]:
edu_ltr_segment_city = df_education\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
edu_ltr_segment_city['% city threshold'] =  (edu_ltr_segment_city['customers']*100.00/edu_ltr_segment_city.customers.sum()).round(2)
edu_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,8496,22.74
1,LTR 0,1275,3.41
2,PHH,27576,73.82
3,UNKNOWN,11,0.03


In [53]:
df1 = df_education[~df_education['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
edu_total_customers = df_education.customer_id.nunique()
df2 = pd.merge(df1,edu_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/edu_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                       \
app_name       aakash brainly    byju caclubindia chegg study colegeduniya   
ltr_segment                                                                  
HH               62.0   823.0  1286.0         1.0        29.0          5.0   
LTR 0             7.0   111.0   233.0         NaN         6.0          3.0   
PHH             110.0  2948.0  3362.0        13.0        85.0         28.0   

                                                                         \
app_name    collegedekho course hero coursera  diksha doubtnut duolingo   
ltr_segment                                                               
HH                   1.0         1.0    233.0   416.0    790.0   1437.0   
LTR 0                NaN         2.0     38.0    58.0    125.0    195.0   
PHH                  3.0         7.0   1259.0  1004.0   1632.0   4703.0   

                                                                            \
app_name    e-pg pathshala    edx embibe fiitjee geeks for geeks goodreads   
ltr_segment                                                                  
HH                   121.0   26.0   31.0     NaN            67.0      23.0   
LTR 0                 11.0    5.0    1.0     NaN             9.0       3.0   
PHH                  458.0  174.0   86.0     1.0           271.0     177.0   

                                                           \
app_name    google classroom ignou e-content khan academy   
ltr_segment                                                 
HH                    1400.0            45.0         22.0   
LTR 0                  208.0             4.0          6.0   
PHH                   5542.0           219.0         82.0   

                                                                    \
app_name    microsoft math solver moodle my study life mycbseguide   
ltr_segment                                                          
HH                           28.0   22.0           1.0        75.0   
LTR 0                         3.0    2.0           NaN        10.0   
PHH                          87.0  119.0           3.0       276.0   

                                                                       \
app_name    ncert books photomath physics wallah pocket shiksha mitra   
ltr_segment                                                             
HH                162.0      47.0         1640.0    6.0           6.0   
LTR 0              16.0       6.0          254.0    NaN           3.0   
PHH               609.0     161.0         4463.0   55.0          15.0   

                                                                          \
app_name    shiksha.com simplilearn stack overflow  swayam toppr   udemy   
ltr_segment                                                                
HH                 10.0        41.0            NaN   346.0   1.0   402.0   
LTR 0               NaN         9.0            NaN    49.0   NaN    51.0   
PHH                26.0       297.0            3.0  1066.0   4.0  2590.0   

                                                    
app_name    unacademy vedantu vidyakul whitehat jr  
ltr_segment                                         
HH             1234.0    44.0      7.0        15.0  
LTR 0           175.0     3.0      2.0         4.0  
PHH            4628.0   109.0     10.0        59.0

In [54]:
# print('% Distribution of customers across individual apps.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [55]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual segment.


customers %                                                      \
app_name         aakash brainly   byju caclubindia chegg study colegeduniya   
ltr_segment                                                                   
HH                 0.73    9.69  15.14        0.01        0.34         0.06   
LTR 0              0.55    8.71  18.27         NaN        0.47         0.24   
PHH                0.40   10.69  12.19        0.05        0.31         0.10   

                                                                        \
app_name    collegedekho course hero coursera diksha doubtnut duolingo   
ltr_segment                                                              
HH                  0.01        0.01     2.74   4.90     9.30    16.91   
LTR 0                NaN        0.16     2.98   4.55     9.80    15.29   
PHH                 0.01        0.03     4.57   3.64     5.92    17.05   

                                                                           \
app_name    e-pg pathshala   edx embibe fiitjee geeks for geeks goodreads   
ltr_segment                                                                 
HH                    1.42  0.31   0.36     NaN            0.79      0.27   
LTR 0                 0.86  0.39   0.08     NaN            0.71      0.24   
PHH                   1.66  0.63   0.31     0.0            0.98      0.64   

                                                           \
app_name    google classroom ignou e-content khan academy   
ltr_segment                                                 
HH                     16.48            0.53         0.26   
LTR 0                  16.31            0.31         0.47   
PHH                    20.10            0.79         0.30   

                                                                    \
app_name    microsoft math solver moodle my study life mycbseguide   
ltr_segment                                                          
HH                           0.33   0.26          0.01        0.88   
LTR 0                        0.24   0.16           NaN        0.78   
PHH                          0.32   0.43          0.01        1.00   

                                                                       \
app_name    ncert books photomath physics wallah pocket shiksha mitra   
ltr_segment                                                             
HH                 1.91      0.55          19.30   0.07          0.07   
LTR 0              1.25      0.47          19.92    NaN          0.24   
PHH                2.21      0.58          16.18   0.20          0.05   

                                                                       \
app_name    shiksha.com simplilearn stack overflow swayam toppr udemy   
ltr_segment                                                             
HH                 0.12        0.48            NaN   4.07  0.01  4.73   
LTR 0               NaN        0.71            NaN   3.84   NaN  4.00   
PHH                0.09        1.08           0.01   3.87  0.01  9.39   

                                                    
app_name    unacademy vedantu vidyakul whitehat jr  
ltr_segment                                         
HH              14.52    0.52     0.08        0.18  
LTR 0           13.73    0.24     0.16        0.31  
PHH             16.78    0.40     0.04        0.21

In [56]:
df1 = df_education[~df_education['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name_tag', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name_tag', 'customers'],ascending=[False,True])\
.reset_index()
edu_total_customers = df_education.customer_id.nunique()
df2 = pd.merge(df1,edu_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/edu_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers'])

customers                        
app_name_tag code_education_app secondary_education_app
ltr_segment                                            
HH                         4808                    4889
LTR 0                       768                     686
PHH                       14682                   17137

In [57]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers %'])

% Distribution of customers across individual segment.


customers %                        
app_name_tag code_education_app secondary_education_app
ltr_segment                                            
HH                        56.59                   57.54
LTR 0                     60.24                   53.80
PHH                       53.24                   62.14

### Other

#### Service Affinity

In [58]:
service_affinity_city = df_customer_data\
                        .groupby(['service_affinity'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
service_affinity_city['% city threshold']=(service_affinity_city['customers']*100.00/service_affinity_city.customers.sum()).round(2)
service_affinity_city.sort_values(by=['service_affinity'],ascending=[True])

,service_affinity,customers,% city threshold
0,AUTO_CAB,114,0.05
1,LINK_AUTO,32623,14.81
2,LINK_CAB,88,0.04
3,NO_AFFINITY,438,0.20
4,ONLY_AUTO,61081,27.72
5,ONLY_CAB,336,0.15
6,ONLY_LINK,110773,50.27
7,UNKNOWN,14894,6.76


In [59]:
df1 = df_customer_data_explode[~df_customer_data_explode['service_affinity'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'service_affinity'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,service_affinity_city[['service_affinity','customers']], how= 'left', on='service_affinity')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='service_affinity' , columns ='categories_l2', values =['customers %', 'customers'])

customers %                                               \
categories_l2     Driver_App Educational Finance_Investment Gaming Office   
service_affinity                                                            
AUTO_CAB                5.26       25.44              48.25  17.54  64.91   
LINK_AUTO               7.01       17.25              26.54  26.55  37.53   
LINK_CAB                5.68       10.23              36.36  25.00  59.09   
NO_AFFINITY             6.62       20.09              35.62  21.46  47.03   
ONLY_AUTO               5.34       20.11              25.60  24.17  40.92   
ONLY_CAB                3.57       19.94              49.70  23.51  54.17   
ONLY_LINK               7.94       15.14              28.71  26.06  37.61   

                                      customers              \
categories_l2       Rest Ride haling Driver_App Educational   
service_affinity                                              
AUTO_CAB          100.00       79.82        6.0        29.0   
LINK_AUTO          99.96       63.62     2288.0      5627.0   
LINK_CAB          100.00       75.00        5.0         9.0   
NO_AFFINITY        99.77       68.72       29.0        88.0   
ONLY_AUTO          99.96       64.88     3260.0     12284.0   
ONLY_CAB          100.00       74.40       12.0        67.0   
ONLY_LINK          99.96       58.50     8791.0     16771.0   

                                                                             
categories_l2    Finance_Investment   Gaming   Office      Rest Ride haling  
service_affinity                                                             
AUTO_CAB                       55.0     20.0     74.0     114.0        91.0  
LINK_AUTO                    8658.0   8661.0  12242.0   32611.0     20755.0  
LINK_CAB                       32.0     22.0     52.0      88.0        66.0  
NO_AFFINITY                   156.0     94.0    206.0     437.0       301.0  
ONLY_AUTO                   15637.0  14762.0  24997.0   61059.0     39627.0  
ONLY_CAB                      167.0     79.0    182.0     336.0       250.0  
ONLY_LINK                   31805.0  28870.0  41663.0  110729.0     64804.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be LINK Customers.
    - Driver_App, Finance_Investment
- Whenever an app belongs to the app-categories listed below, customers are more likely to be AUTO Customers.
    - Educational, Office, Ride haling

#### Distance Preference

In [60]:
distance_affinity_city = df_customer_data\
                        .groupby(['distance_preference'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
distance_affinity_city['% city threshold']=(distance_affinity_city['customers']*100.00/distance_affinity_city.customers.sum()).round(2)
distance_affinity_city.sort_values(by=['distance_preference'],ascending=[True])

,distance_preference,customers,% city threshold
0,LONG_DISTANCE,54059,24.53
1,MEDIUM_DISTANCE,52722,23.93
2,SHORT_DISTANCE,52146,23.67
3,UNKNOWN,61420,27.87


In [61]:
df1 = df_customer_data_explode[~df_customer_data_explode['distance_preference'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'distance_preference'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,distance_affinity_city[['distance_preference','customers']], how= 'left', on='distance_preference')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='distance_preference' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2        Driver_App Educational Finance_Investment Gaming Office   
distance_preference                                                            
LONG_DISTANCE              8.50       15.20              27.86  26.23  37.53   
MEDIUM_DISTANCE            6.29       16.91              27.74  24.45  39.09   
SHORT_DISTANCE             5.30       19.03              26.89  24.67  39.78   

                                        customers              \
categories_l2         Rest Ride haling Driver_App Educational   
distance_preference                                             
LONG_DISTANCE        99.96       58.85     4593.0      8217.0   
MEDIUM_DISTANCE      99.96       59.83     3314.0      8917.0   
SHORT_DISTANCE       99.97       64.15     2765.0      9924.0   

                                                                               
categories_l2       Finance_Investment   Gaming   Office     Rest Ride haling  
distance_preference                                                            
LONG_DISTANCE                  15063.0  14180.0  20286.0  54037.0     31813.0  
MEDIUM_DISTANCE                14625.0  12892.0  20608.0  52700.0     31543.0  
SHORT_DISTANCE                 14021.0  12864.0  20742.0  52128.0     33450.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to do LONG Distance rides.
    - Driver_App, Gaming
   
- Whenever an app belongs to the app-categories listed below, customers are more likely to do MEDIUM Distance rides.
    - Finance_Investment
   
- Whenever an app belongs to the app-categories listed below, customers are more likely to do SHORT Distance rides.
    - Educational, Office

#### Gender

In [62]:
gender_city = df_customer_data\
                        .groupby(['gender'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
gender_city['% city threshold']=(gender_city['customers']*100.00/gender_city.customers.sum()).round(2)
gender_city.sort_values(by=['gender'],ascending=[True])

,gender,customers,% city threshold
0,FEMALE,51706,23.47
1,MALE,162319,73.67
2,OTHERS,618,0.28
3,UNKNOWN,5704,2.59


In [63]:
df1 = df_customer_data_explode[~df_customer_data_explode['gender'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'gender'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,gender_city[['gender','customers']], how= 'left', on='gender')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='gender' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   
gender                                                                   
FEMALE               3.21       22.05              17.76  25.44  42.27   
MALE                 8.37       15.28              30.59  25.52  37.34   
OTHERS               7.28       14.56              14.24  27.67  26.38   

                                   customers                                 \
categories_l2    Rest Ride haling Driver_App Educational Finance_Investment   
gender                                                                        
FEMALE          99.96       68.14     1659.0     11402.0             9181.0   
MALE            99.96       58.32    13578.0     24803.0            49649.0   
OTHERS         100.00       48.71       45.0        90.0               88.0   

                                                       
categories_l2   Gaming   Office      Rest Ride haling  
gender                                                 
FEMALE         13153.0  21857.0   51687.0     35235.0  
MALE           41423.0  60605.0  162255.0     94670.0  
OTHERS           171.0    163.0     618.0       301.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be MALE.
    - Driver_App, Finance_Investment
- Whenever an app belongs to the app-categories listed below, customers are more likely to be FEMALE.
    - Educational

#### Income Segment

In [64]:
income_segment_city = df_customer_data\
                        .groupby(['income_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
income_segment_city['% city threshold']=(income_segment_city['customers']*100.00/income_segment_city.customers.sum()).round(2)
income_segment_city.sort_values(by=['income_segment'],ascending=[True])

,income_segment,customers,% city threshold
0,HIGH_INCOME,74250,33.70
1,LOW_INCOME,16742,7.60
2,MEDIUM_INCOME,91956,41.73
3,UNKNOWN,37399,16.97


In [65]:
df1 = df_customer_data_explode[~df_customer_data_explode['income_segment'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'income_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,income_segment_city[['income_segment','customers']], how= 'left', on='income_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='income_segment' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2   Driver_App Educational Finance_Investment Gaming Office   
income_segment                                                            
HIGH_INCOME           7.49       19.97              36.06  30.08  46.93   
LOW_INCOME            4.84       10.38              12.45  14.66  21.14   
MEDIUM_INCOME         7.06       17.38              23.91  24.51  37.65   

                                   customers                                 \
categories_l2    Rest Ride haling Driver_App Educational Finance_Investment   
income_segment                                                                
HIGH_INCOME     99.99       70.86     5558.0     14825.0            26777.0   
LOW_INCOME      99.81       40.01      810.0      1738.0             2085.0   
MEDIUM_INCOME   99.97       58.60     6491.0     15986.0            21990.0   

                                                       
categories_l2    Gaming   Office     Rest Ride haling  
income_segment                                         
HIGH_INCOME     22335.0  34845.0  74242.0     52617.0  
LOW_INCOME       2454.0   3540.0  16711.0      6698.0  
MEDIUM_INCOME   22537.0  34623.0  91927.0     53886.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be HIGH_INCOME.
    - Office, Educational, Finance_Investment, Gaming, Ride haling
- Whenever an app belongs to the app-categories listed below, customers are more likely to be MEDIUM_INCOME.
    - Driver_App

#### Price Sensitivity

In [66]:
ps_city = df_customer_data\
                        .groupby(['price_sensitivity'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
ps_city['% city threshold']=(ps_city['customers']*100.00/ps_city.customers.sum()).round(2)
ps_city.sort_values(by=['price_sensitivity'],ascending=[True])

,price_sensitivity,customers,% city threshold
0,NPS,57814,26.24
1,PS,55573,25.22
2,UNKNOWN,106960,48.54


In [67]:
df1 = df_customer_data_explode[~df_customer_data_explode['price_sensitivity'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'price_sensitivity'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,ps_city[['price_sensitivity','customers']], how= 'left', on='price_sensitivity')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='price_sensitivity' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2      Driver_App Educational Finance_Investment Gaming Office   
price_sensitivity                                                            
NPS                      7.74       16.11              27.07  26.86  38.29   
PS                       6.67       17.95              27.35  26.03  39.75   

                                      customers              \
categories_l2       Rest Ride haling Driver_App Educational   
price_sensitivity                                             
NPS                99.96       64.49     4476.0      9311.0   
PS                 99.98       66.94     3709.0      9978.0   

                                                                             
categories_l2     Finance_Investment   Gaming   Office     Rest Ride haling  
price_sensitivity                                                            
NPS                          15653.0  15531.0  22139.0  57790.0     37284.0  
PS                           15200.0  14468.0  22090.0  55561.0     37199.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be Price Sensitivity.
    - Driver_App
- Whenever an app belongs to the app-categories listed below, customers are less likely to be Price Sensitivity. (Less confidence - Uninterpretable/Hard to interpretable using Price Sensitivity)
    - Office, Educational, Finance_Investment, Ride haling, Gaming

#### RPC

In [68]:
rpc_bucket_city = df_customer_data\
                        .groupby(['rpc_bucket'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
rpc_bucket_city['% city threshold']=(rpc_bucket_city['customers']*100.00/rpc_bucket_city.customers.sum()).round(2)
rpc_bucket_city.sort_values(by=['rpc_bucket'],ascending=[True])

,rpc_bucket,customers,% city threshold
0,a. ZERO,49265,22.36
1,b. LOW,97702,44.34
2,c. MEDIUM,50753,23.03
3,d. HIGH,22625,10.27


In [69]:
df1 = df_customer_data_explode[~df_customer_data_explode['rpc_bucket'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'rpc_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,rpc_bucket_city[['rpc_bucket','customers']], how= 'left', on='rpc_bucket')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='rpc_bucket' , columns ='categories_l2', values =['customers %','customers'])

customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
rpc_bucket                                                                      
a. ZERO              8.56       17.66              26.57  26.34  36.45  99.97   
b. LOW               6.97       16.53              27.41  25.43  37.57  99.96   
c. MEDIUM            6.31       17.01              27.63  25.32  39.75  99.96   
d. HIGH              5.66       17.12              28.10  24.92  43.81  99.96   

                           customers                                          \
categories_l2 Ride haling Driver_App Educational Finance_Investment   Gaming   
rpc_bucket                                                                     
a. ZERO             62.07     4219.0      8698.0            13090.0  12975.0   
b. LOW              58.30     6808.0     16155.0            26781.0  24845.0   
c. MEDIUM           60.25     3201.0      8632.0            14023.0  12850.0   
d. HIGH             68.65     1281.0      3873.0             6358.0   5638.0   

                                             
categories_l2   Office     Rest Ride haling  
rpc_bucket                                   
a. ZERO        17959.0  49248.0     30578.0  
b. LOW         36709.0  97659.0     56964.0  
c. MEDIUM      20176.0  50735.0     30579.0  
d. HIGH         9912.0  22615.0     15532.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers having Less RPC
    - Driver_App
- Whenever an app belongs to the app-categories listed below, customers having Good RPC
    - Educational, Finance_Investment, Office, Ride haling

#### App Count Bucket

In [70]:
app_count_bucket_city = df_customer_data\
                        .groupby(['app_count_bucket'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
app_count_bucket_city['% city threshold']=(app_count_bucket_city['customers']*100.00/app_count_bucket_city.customers.sum()).round(2)
app_count_bucket_city.sort_values(by=['app_count_bucket'],ascending=[True])

,app_count_bucket,customers,% city threshold
0,1-5,205532,93.28
1,11-15,2616,1.19
2,6-10,11234,5.10
3,Above 15,965,0.44


In [71]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'app_count_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'app_count_bucket'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='app_count_bucket' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2    Driver_App Educational Finance_Investment Gaming Office   
app_count_bucket                                                           
1-5                   94.78       93.17              92.89  93.51  92.06   
11-15                  0.93        1.18               1.35   1.12   1.49   
6-10                   4.02        5.23               5.32   4.99   5.90   
Above 15               0.26        0.42               0.44   0.39   0.55   

                                     
categories_l2      Rest Ride haling  
app_count_bucket                     
1-5               93.28       92.16  
11-15              1.19        1.46  
6-10               5.10        5.83  
Above 15           0.44        0.55

In [72]:
df_customer_data.app_count.describe()

count    220347.000000
mean         15.033456
std           7.706327
min           1.000000
25%           9.000000
50%          14.000000
75%          19.000000
max          72.000000
Name: app_count, dtype: float64

#### Category Count Bucket

In [73]:
cat_count_bucket_city = df_customer_data\
                        .groupby(['category_count_bucket'])\
                        .agg(customers = ('customer_id','nunique'),)\
                        .reset_index()
cat_count_bucket_city['% city threshold']=(cat_count_bucket_city['customers']*100.00/cat_count_bucket_city.customers.sum()).round(2)
cat_count_bucket_city.sort_values(by=['category_count_bucket'],ascending=[True])

,category_count_bucket,customers,% city threshold
0,1,37135,16.85
1,2,62145,28.20
2,3,60543,27.48
3,Above 3,60524,27.47


In [74]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'category_count_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'category_count_bucket'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='category_count_bucket' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2         Driver_App Educational Finance_Investment Gaming Office   
category_count_bucket                                                           
1                           0.01        0.02               0.01   0.01   0.01   
2                          12.89        9.60              10.91  16.16  12.25   
3                          29.35       24.66              27.22  31.19  31.74   
Above 3                    57.75       65.71              61.86  52.64  55.99   

                                          
categories_l2           Rest Ride haling  
category_count_bucket                     
1                      16.83        0.03  
2                      28.21       22.83  
3                      27.49       34.76  
Above 3                27.48       42.38

In [75]:
df_temp = df_customer_data[df_customer_data['category_count_bucket'].isin(['2-3'])]\
.groupby(['customer_id'])\
.agg({'categories_l2' : set})\
.reset_index()
df_temp[['categories_l2']]

,categories_l2


In [76]:
df_customer_data.category_count.describe()

count    220347.000000
mean          2.759729
std           1.232630
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max           7.000000
Name: category_count, dtype: float64

#### AO-NET Funnel

In [77]:
df_customer_data['city'] = 'Jaipur, Android'
city_funnel = df_customer_data\
                        .groupby(['city'])\
                        .agg(fe_count = ('fe_count','sum'),
                             rr_count = ('rr_count','sum'),
                             net_count = ('net_count','sum')
                            )\
                        .reset_index()

city_funnel['% fe2rr']=(city_funnel['rr_count']*100.00/city_funnel['fe_count']).round(2)
city_funnel['% g2n']=(city_funnel['net_count']*100.00/city_funnel['rr_count']).round(2)
city_funnel['% fe2net']=(city_funnel['net_count']*100.00/city_funnel['fe_count']).round(2)
city_funnel[['city','% fe2rr','% g2n','% fe2net']].T

,0
city,"Jaipur, Android"
% fe2rr,40.5
% g2n,56.34
% fe2net,22.82


In [78]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y'])\
.agg(customers = ('customer_id','nunique'),
     fe_count = ('fe_count','sum'),
     rr_count = ('rr_count','sum'),
     net_count = ('net_count','sum')
    )\
.sort_values(by=['categories_l2_y'],ascending=[False])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['% fe2rr']=(df2['rr_count']*100.00/df2['fe_count']).round(2)
df2['% g2n']=(df2['net_count']*100.00/df2['rr_count']).round(2)
df2['% fe2net']=(df2['net_count']*100.00/df2['fe_count']).round(2)
df2[['categories_l2','% fe2rr','% g2n','% fe2net']].T

,0,1,2,3,4,5,6
categories_l2,Rest,Ride haling,Office,Finance_Investment,Gaming,Educational,Driver_App
% fe2rr,40.5,42.22,41.75,40.82,41.3,40.68,40.51
% g2n,56.34,55.98,57.99,58.01,55.63,55.48,50.84
% fe2net,22.82,23.64,24.21,23.68,22.98,22.57,20.6
